In [19]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from dataclasses import dataclass
from typing import List, Tuple
import wandb

In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

In [15]:
wandb.login(key=secret_value_0)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bhalaniakshat (bhalaniakshat-dwarkadas-j-sanghvi-college-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [22]:
CONFIG = {
    "env_id": "CartPole-v1",
    "hidden_size": 128,
    "batch_size": 16,
    "percentile": 70,
    "lr": 0.01,
    "solve_bound": 450.0
}

In [23]:
@dataclass
class EpisodeStep:
    observation: np.ndarray
    action: int

In [24]:
@dataclass
class Episode:
    reward: float
    steps: List[EpisodeStep]

In [25]:
class Net(nn.Module):
    def __init__(self, obs_size: int, hidden_size: int, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [26]:
@torch.inference_mode()
def iterate_batches(env: gym.Env, net: nn.Module, device: str):
    batch: List[Episode] = []
    episode_reward = 0.0
    episode_steps = []
    obs, _ = env.reset()
    
    while True:
        obs_v = torch.as_tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits = net(obs_v)
        probs = torch.softmax(logits, dim=1)
        m = Categorical(probs)
        action = m.sample().item()
        next_obs, reward, terminated, truncated, _ = env.step(action)
        episode_reward += reward
        episode_steps.append(EpisodeStep(observation=obs, action=action))
        if terminated or truncated:
            batch.append(Episode(reward=episode_reward, steps=episode_steps))
            episode_reward = 0.0
            episode_steps = []
            next_obs, _ = env.reset()
            if len(batch) == CONFIG["batch_size"]:
                yield batch
                batch = []
        obs = next_obs

In [27]:
def filter_batch(batch: List[Episode], percentile: float, device: str):
    rewards = [s.reward for s in batch]
    reward_bound = np.percentile(rewards, percentile)
    reward_mean = float(np.mean(rewards))
    train_obs = []
    train_act = []
    for example in batch:
        if example.reward < reward_bound:
            continue
        train_obs.extend([step.observation for step in example.steps])
        train_act.extend([step.action for step in example.steps])
    train_obs_v = torch.as_tensor(np.array(train_obs), dtype=torch.float32, device=device)
    train_act_v = torch.as_tensor(train_act, dtype=torch.long, device=device)
    return train_obs_v, train_act_v, reward_bound, reward_mean

In [28]:
run = wandb.init(project="modern-cross-entropy", config=CONFIG)
device = "cuda" if torch.cuda.is_available() else "cpu"
env = gym.make(CONFIG["env_id"])
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n
net = Net(obs_size, CONFIG["hidden_size"], n_actions).to(device)
objective = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=CONFIG["lr"])

In [31]:
for iter_no, batch in enumerate(iterate_batches(env, net, device)):
        obs_v, acts_v, reward_b, reward_m = filter_batch(batch, CONFIG["percentile"], device)
        
        optimizer.zero_grad()
        action_scores_v = net(obs_v)
        loss_v = objective(action_scores_v, acts_v)
        loss_v.backward()
        optimizer.step()

        # Log to CLI and WandB
        print(f"{iter_no}: loss={loss_v.item():.3f}, reward_mean={reward_m:.1f}, reward_bound={reward_b:.1f}")
        
        wandb.log({
            "loss": loss_v.item(),
            "reward_mean": reward_m,
            "reward_bound": reward_b,
            "iteration": iter_no
        })

        if reward_m > CONFIG["solve_bound"]:
            print(f"Solved in {iter_no} iterations!")
            break

wandb.finish()
env.close()

0: loss=0.677, reward_mean=17.3, reward_bound=18.0
1: loss=0.692, reward_mean=14.6, reward_bound=16.0
2: loss=0.686, reward_mean=21.4, reward_bound=24.5
3: loss=0.695, reward_mean=17.3, reward_bound=17.5
4: loss=0.703, reward_mean=22.1, reward_bound=25.0
5: loss=0.677, reward_mean=20.4, reward_bound=20.0
6: loss=0.667, reward_mean=26.9, reward_bound=34.0
7: loss=0.658, reward_mean=26.9, reward_bound=26.0
8: loss=0.651, reward_mean=33.5, reward_bound=32.0
9: loss=0.635, reward_mean=39.6, reward_bound=45.0
10: loss=0.637, reward_mean=39.6, reward_bound=47.0
11: loss=0.632, reward_mean=48.9, reward_bound=62.5
12: loss=0.628, reward_mean=50.6, reward_bound=55.5
13: loss=0.604, reward_mean=46.2, reward_bound=62.0
14: loss=0.613, reward_mean=52.7, reward_bound=63.5
15: loss=0.601, reward_mean=38.3, reward_bound=46.0
16: loss=0.597, reward_mean=45.9, reward_bound=45.5
17: loss=0.574, reward_mean=61.4, reward_bound=69.5
18: loss=0.590, reward_mean=79.2, reward_bound=88.5
19: loss=0.578, reward

iteration,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
loss,▇█▇██▇▇▆▆▆▆▅▅▅▄▄▄▃▄▂▂▃▂▂▂▂▁▃▂▁▂▂▂▁▁▁▁▁▁▁
reward_bound,▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▅▇▆▅▇█
reward_mean,▁▁▁▁▁▁▁▁▁▁▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▅▆▆▅▆█
iteration,47
loss,0.49736
reward_bound,500
reward_mean,451.125
